In [1]:
%matplotlib inline
import control
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.signal

sns.set_style("whitegrid")

Cruise Control: System Modeling


Key MATLAB commands used in this tutorial are:

<http://www.mathworks.com/help/toolbox/control/ref/ss.html |ss|> , 

<http://www.mathworks.com/help/toolbox/control/ref/tf.html |tf|>


Physical setup


Automatic *cruise control* is an excellent example of a feedback control 

system found in many modern vehicles. The purpose of the cruise control 

system is to maintain a constant vehicle speed despite external *disturbances*, 

such as changes in wind or road grade. This is accomplished by 

measuring the vehicle speed, comparing it to the desired or *reference* speed, 

and automatically adjusting the throttle according to a *control law*.


![cruise_control_schematic.png](figures/cruise_control_schematic.png)


We consider here a simple model of the vehicle dynamics, shown in the

free-body diagram (FBD) above.  The vehicle, of mass m, is acted on by a

control force, u. The force u represents the force generated at the

road/tire interface. For this simplified model we will assume that we can

control this force directly and will neglect the dynamics of the

powertrain, tires, etc., that go into generating the force. The resistive 

forces, bv, due to rolling resistance and wind 

drag, are assumed to vary linearly with the vehicle velocity, v, and act 

in the direction opposite the vehicle's motion.  

System equations


With these assumptions we are left with a

< ?example=Introduction&section=SystemAnalysis#8 first-order>

mass-damper system. Summing forces in the x-direction and applying Newton's 2nd law, we

arrive at the following system equation:


$$

m \dot{v} + b v = u 

$$


Since we are interested in controlling the speed of the vehicle, the output 

equation is chosen as follows


Modelica Model

This system has also been implemented in Modelica. You can use the Modelica model
to simulate the system and compare results with the Python Control Systems Library.

To use the Modelica model, you need OpenModelica installed (see Modelica/README.md
for installation instructions) and the OMPython package.


In [2]:
# Optional: Modelica simulation using OMPython
# Uncomment the following code if you have OpenModelica and OMPython installed
# 
# try:
#     from OMPython import ModelicaSystem
#     import os
#     
#     # Get the path to the Modelica model
#     modelica_path = os.path.join(os.path.dirname(os.getcwd()), 
#                                   'Modelica', 'UMichControls', 
#                                   'CruiseControl', 'CruiseControl_System.mo')
#     
#     # Create model instance
#     model = ModelicaSystem(modelica_path, "CruiseControl_System")
#     
#     # Set parameters
#     model.setParameters("m=1000", "b=50")
#     
#     # Set simulation options
#     model.setSimulationOptions("startTime=0", "stopTime=10", "stepSize=0.01")
#     
#     # Set input (step input of 500 N)
#     model.setInputs("u=500")
#     
#     # Simulate
#     model.simulate()
#     
#     # Get results
#     results = model.getSolutions()
#     time_modelica = results['time']
#     velocity_modelica = results['v']
#     
#     # Plot comparison
#     plt.figure(figsize=(10, 6))
#     plt.plot(time_modelica, velocity_modelica, 'r--', label='Modelica', linewidth=2)
#     plt.xlabel('Time (s)')
#     plt.ylabel('Velocity (m/s)')
#     plt.title('Cruise Control System: Modelica vs Python')
#     plt.legend()
#     plt.grid(True)
#     plt.show()
#     
#     print("Modelica simulation completed successfully!")
# except ImportError:
#     print("OMPython not available. Install with: pip install OMPython")
#     print("Also ensure OpenModelica is installed (see Modelica/README.md)")
# except Exception as e:
#     print(f"Modelica simulation error: {e}")
#     print("This is optional - the Python implementation above works independently")


$$ 

y = v

$$


System parameters


For this example, let's assume that the parameters of the system are:


(m)   vehicle mass          1000 kg


(b)   damping coefficient   50 N.s/m


State-space model


First-order systems have only a single energy storage mode, in this case the

kinetic energy of the car, and therefore only one state variable is

needed, the velocity.  The state-space representation is therefore:


$$

\dot{\mathbf{x}}=[\dot{v}]=\left[\frac{-b}{m}\right][v]+\left[\frac{1}{m}\right][u]

$$


$$

y=[1][v]

$$


We enter this state-space model into MATLAB using the following

commands:


In [3]:

m = 1000
b = 50

A = -b/m
B = 1/m
C = 1
D = 0

cruise_ss = control.StateSpace(A,B,C,D)


Transfer function model


Taking the Laplace transform of the governing differential equation and

assuming zero initial conditions, we find the transfer function of the

cruise control system to be:


$$

P(s) = \frac{V(s)}{U(s)} = \frac{1}{ms+b}  \qquad  [ \frac{m/s}{N} ]

$$


We enter the transfer function model into MATLAB using the following

commands:


In [4]:

s = control.tf('s')
P_cruise = 1/(m*s+b)